## Evaluate a custom Presidio Analyzer using the Presidio Evaluator framework

This notebook demonstrates how to evaluate a Presidio instance using the presidio-evaluator framework. It builds upon [notebook 4](4_Evaluate_Presidio_Analyzer.ipynb), customising the `PresidioAnalyzer` instance for better accuracy. For more on customising Presidio, see the [Presidio Analyzer documentation](https://microsoft.github.io/presidio/analyzer/).

Steps:
1. Load dataset
2. Dataset statistics
3. Define the custom Presidio Analyzer
4. Run predictions
5. Review and adjust entity mapping
6. Evaluate
7. Results and error analysis

In [1]:
# install presidio evaluator via pip if not yet installed

#!pip install presidio-evaluator
#!pip install "presidio-analyzer[transformers]"

In [2]:
import json
import warnings
from collections import Counter
from pathlib import Path
from pprint import pprint

warnings.filterwarnings("ignore")

import pandas as pd
from presidio_analyzer import (
    AnalyzerEngine,
    Pattern,
    PatternRecognizer,
    RecognizerRegistry,
)
from presidio_analyzer.context_aware_enhancers import LemmaContextAwareEnhancer
from presidio_analyzer.nlp_engine import NerModelConfiguration, TransformersNlpEngine

from presidio_evaluator import InputSample
from presidio_evaluator.entity_mapping import CanonicalMapper
from presidio_evaluator.evaluation import ModelError, Plotter, SpanEvaluator
from presidio_evaluator.experiment_tracking import get_experiment_tracker
from presidio_evaluator.models import PresidioAnalyzerWrapper

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2
%matplotlib inline

## 1. Load dataset from file

In [3]:
dataset_name = "synth_dataset_v2.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd().parent, "data", dataset_name))

print(len(dataset))

tokenizing input:   0%|          | 1/1500 [00:00<04:54,  5.08it/s]

loading model en_core_web_sm


tokenizing input: 100%|██████████| 1500/1500 [00:05<00:00, 292.45it/s]

1500


This dataset was auto generated. See more info here [Synthetic data generation](1_Generate_data.ipynb).

In [4]:
def get_entity_counts(dataset: list[InputSample]) -> dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter

## 2. Simple dataset statistics

In [5]:
entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print(
    "\nMin and max number of tokens in dataset: "
    f"Min: {min([len(sample.tokens) for sample in dataset])}, "
    f"Max: {max([len(sample.tokens) for sample in dataset])}"
)

print(
    f"Min and max sentence length in dataset: "
    f"Min: {min([len(sample.full_text) for sample in dataset])}, "
    f"Max: {max([len(sample.full_text) for sample in dataset])}"
)

Count per entity:
[('O', 19626), ('STREET_ADDRESS', 3071), ('PERSON', 1369), ('GPE', 521),
 ('ORGANIZATION', 504), ('PHONE_NUMBER', 350), ('DATE_TIME', 219),
 ('TITLE', 142), ('CREDIT_CARD', 136), ('US_SSN', 80), ('AGE', 74), ('NRP', 55),
 ('ZIP_CODE', 50), ('EMAIL_ADDRESS', 49), ('DOMAIN_NAME', 37),
 ('IP_ADDRESS', 22), ('IBAN_CODE', 21), ('US_DRIVER_LICENSE', 9)]

Min and max number of tokens in dataset: Min: 3, Max: 78
Min and max sentence length in dataset: Min: 9, Max: 407


## 3. Define the AnalyzerEngine object 
In this case, we customize the AnalyzerEngine to use a different NER model, some custom recognizers and the context aware enhancer.

### 3.1 Set up the NlpEngine and Hugging Face NER Recognizer
The NLP engine uses spaCy for text processing (lemmas, tokens, etc.). The Hugging Face model is added as a direct recognizer (HuggingFaceNerRecognizer) instead of being part of the NLP engine. This approach gives better control over model predictions.

In [6]:
# Map OpenMed model entities to Presidio entities
mapping = {
    # Personal Info - Names & Demographics
    "first_name": "PERSON",
    "last_name": "PERSON",
    "age": "AGE",
    "gender": "GENDER",
    "date_of_birth": "DATE_TIME",
    "blood_type": "BLOOD_TYPE",
    "occupation": "OCCUPATION",
    "education_level": "EDUCATION_LEVEL",
    "employment_status": "EMPLOYMENT_STATUS",
    "language": "LOCATION",
    "race_ethnicity": "ETHNICITY",
    "sexuality": "SEXUALITY",
    "political_view": "ORGANIZATION",
    "religious_belief": "ORGANIZATION",
    "biometric_identifier": "ID",
    "pin": "ID",
    # Identifiers (20+ types)
    "account_number": "ID",
    "customer_id": "ID",
    "employee_id": "ID",
    "unique_id": "ID",
    "bank_routing_number": "US_BANK_NUMBER",
    "swift_bic": "SWIFT_CODE",
    "certificate_license_number": "MEDICAL_LICENSE",
    "medical_record_number": "MEDICAL_LICENSE",
    "health_plan_beneficiary_number": "ID",
    "credit_debit_card": "CREDIT_CARD",
    "cvv": "CVV",
    "ssn": "US_SSN",
    "tax_id": "ID",
    "license_plate": "LICENSE_PLATE",
    "vehicle_identifier": "ID",
    "mac_address": "ID",
    "device_identifier": "ID",
    # Authentication & Security
    "password": "PASSWORD",
    "user_name": "USER_NAME",
    # Contact Info (4 types)
    "email": "EMAIL_ADDRESS",
    "phone_number": "PHONE_NUMBER",
    "fax_number": "PHONE_NUMBER",
    "url": "URL",
    # Location (6 types)
    "city": "LOCATION",
    "country": "LOCATION",
    "county": "LOCATION",
    "state": "LOCATION",
    "street_address": "LOCATION",
    "coordinate": "LOCATION",
    "postcode": "ZIP_CODE",
    # Network Info
    "ipv4": "IP_ADDRESS",
    "ipv6": "IP_ADDRESS",
    # Temporal (3 types)
    "date": "DATE_TIME",
    "date_time": "DATE_TIME",
    "time": "DATE_TIME",
    # Organization (1 type)
    "company_name": "ORGANIZATION",
}

models = [
    {
        "lang_code": "en",
        "model_name": {
            "spacy": "en_core_web_sm",  # use a small spaCy model for lemmas, tokens etc.
            "transformers": "OpenMed/OpenMed-PII-BioClinicalModern-Large-395M-v1",
        },
    }
]

ner_config = NerModelConfiguration(
    model_to_presidio_entity_mapping=mapping, aggregation_strategy="simple"
)

# Create the NLP engine without entity mapping (will be handled by CanonicalMapper)
nlp_engine = TransformersNlpEngine(models=models, ner_model_configuration=ner_config)
nlp_engine.load()

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Device set to use cpu


### 3.2 Set up the relevant recognizers
Add and remove recognizers to fit the dataset in hand. 
Removing all the recognizers that don't map to entities in our dataset.

In [7]:
def get_titles_recognizer():
    titles_recognizer = PatternRecognizer(
        deny_list=["Mr.", "Mrs.", "Ms.", "Miss", "Dr.", "Prof."],
        supported_entity="TITLE",
        name="TitlesRecognizer",
    )
    return titles_recognizer


def get_years_recognizer():
    years_recognizer = PatternRecognizer(
        patterns=[Pattern("YEAR", r"\b(19|20)\d{2}\b", score=0.1)],
        supported_entity="DATE_TIME",
        name="YearsRecognizer",
        context=["year", "at", "date", "in", "on"],
    )
    return years_recognizer


def get_age_recognizer():
    weak_regex = r"\b(110|[1-9]?[0-9])\b"
    age_pattern = Pattern(name="age (very weak)", regex=weak_regex, score=0.01)
    age_recognizer = PatternRecognizer(
        supported_entity="AGE",
        patterns=[age_pattern],
        name="AgeRecognizer",
        context=["month", "old", "turn", "age", "y/o"],
    )
    return age_recognizer


# Create Recognizer Registry
registry = RecognizerRegistry()
registry.load_predefined_recognizers(nlp_engine=nlp_engine)

# Add custom pattern recognizers
registry.add_recognizer(get_titles_recognizer())
registry.add_recognizer(get_years_recognizer())
registry.add_recognizer(get_age_recognizer())

# Remove unnecessary recognizers from presidio
unnecessary = [
    "NhsRecognizer",
    "UkNinoRecognizer",
    "SgFinRecognizer",
    "AuAbnRecognizer",
    "AuAcnRecognizer",
    "AuTfnRecognizer",
    "AuMedicareRecognizer",
    "InPanRecognizer",
    "InAadhaarRecognizer",
    "InVehicleRegistrationRecognizer",
    "InPassportRecognizer",
    "InVoterRecognizer",
    "UsLicenseRecognizer",
    "CryptoRecognizer",
    "SpacyRecognizer",
]

for rec in unnecessary:
    registry.remove_recognizer(rec)

### 3.3 Configure the context mechanism
Configure the `LemmaContextAawareEnhancer` which uses surrounding words to increase confidence in detection

In [8]:
# Set up the context aware enhancer
context_enhancer = LemmaContextAwareEnhancer(
    context_prefix_count=10, context_suffix_count=10
)

### 3.4 Create the AnalyzerEngine object

In [9]:
# Set up the engine, loads the NLP module (spaCy model by default)
# and other PII recognizers
analyzer_engine = AnalyzerEngine(
    nlp_engine=nlp_engine,
    context_aware_enhancer=context_enhancer,
    registry=registry,
    default_score_threshold=0.3,
)

# analyzer_engine = AnalyzerEngine()

pprint("Supported entities for English:")
pprint(analyzer_engine.get_supported_entities("en"), compact=True)

print("\nLoaded recognizers for English:")
pprint(
    [
        rec.name
        for rec in analyzer_engine.registry.get_recognizers("en", all_fields=True)
    ],
    compact=True,
)

print("\nLoaded Context Aware Enhancer:")
print(analyzer_engine.context_aware_enhancer.__class__.__name__)
pprint(json.dumps(analyzer_engine.context_aware_enhancer.__dict__), compact=True)


print("\nLoaded NER models:")
pprint(analyzer_engine.nlp_engine.models)

'Supported entities for English:'
['EDUCATION_LEVEL', 'US_SSN', 'US_ITIN', 'USER_NAME', 'MEDICAL_LICENSE',
 'IBAN_CODE', 'LICENSE_PLATE', 'SWIFT_CODE', 'GENDER', 'BLOOD_TYPE',
 'IP_ADDRESS', 'PERSON', 'ZIP_CODE', 'ORGANIZATION', 'SEXUALITY', 'CVV',
 'PASSWORD', 'EMAIL_ADDRESS', 'ETHNICITY', 'LOCATION', 'PHONE_NUMBER',
 'EMPLOYMENT_STATUS', 'CREDIT_CARD', 'US_PASSPORT', 'TITLE', 'MAC_ADDRESS',
 'DATE_TIME', 'AGE', 'OCCUPATION', 'US_BANK_NUMBER', 'ID', 'URL']

Loaded recognizers for English:
['CreditCardRecognizer', 'UsBankRecognizer', 'UsItinRecognizer',
 'UsPassportRecognizer', 'UsSsnRecognizer', 'DateRecognizer', 'EmailRecognizer',
 'IbanRecognizer', 'IpRecognizer', 'MedicalLicenseRecognizer',
 'MacAddressRecognizer', 'PhoneRecognizer', 'UrlRecognizer',
 'TransformersRecognizer', 'TitlesRecognizer', 'YearsRecognizer',
 'AgeRecognizer']

Loaded Context Aware Enhancer:
LemmaContextAwareEnhancer
('{"context_similarity_factor": 0.35, "min_score_with_context_similarity": '
 '0.4, "context_

In [10]:
# Test Analyzer
text = "Yesterday in Mt. Sinai AP: Dana Silver, 79 years old female was complaining of stomach pain. Her ID is 154555"
res = analyzer_engine.analyze(text=text, language="en", return_decision_process=True)
for result in res:
    print(
        f"\nEntity: {result.entity_type}, Text: {text[result.start : result.end]}\n\nAnalysis explanation:"
    )
    pprint(result.analysis_explanation)


Entity: PERSON, Text: Dana

Analysis explanation:
{'recognizer': 'TransformersRecognizer', 'pattern_name': None, 'pattern': None, 'original_score': 0.9997214674949646, 'score': 0.9997214674949646, 'textual_explanation': "Identified as PERSON by Transformers's Named Entity Recognition", 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': None, 'regex_flags': None}

Entity: PERSON, Text: Silver

Analysis explanation:
{'recognizer': 'TransformersRecognizer', 'pattern_name': None, 'pattern': None, 'original_score': 0.9988800883293152, 'score': 0.9988800883293152, 'textual_explanation': "Identified as PERSON by Transformers's Named Entity Recognition", 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': None, 'regex_flags': None}

Entity: AGE, Text: 79

Analysis explanation:
{'recognizer': 'TransformersRecognizer', 'pattern_name': None, 'pattern': None, 'original_score': 0.99313884973526, 'score': 0.99313884973526, 'textual_ex

In [ ]:
# Wrap the analyzer
wrapped_analyzer = PresidioAnalyzerWrapper(
    analyzer_engine=analyzer_engine,
    score_threshold=analyzer_engine.default_score_threshold,
    language="en",
)

--------
Entities supported by this Presidio Analyzer instance:
EDUCATION_LEVEL, US_SSN, US_ITIN, USER_NAME, MEDICAL_LICENSE, IBAN_CODE, LICENSE_PLATE, SWIFT_CODE, GENDER, BLOOD_TYPE, IP_ADDRESS, PERSON, ZIP_CODE, ORGANIZATION, SEXUALITY, CVV, PASSWORD, EMAIL_ADDRESS, ETHNICITY, LOCATION, PHONE_NUMBER, EMPLOYMENT_STATUS, CREDIT_CARD, US_PASSPORT, TITLE, MAC_ADDRESS, DATE_TIME, AGE, OCCUPATION, US_BANK_NUMBER, ID, URL


## 4. Run predictions

Run the model on the full dataset. Returns a 5-column DataFrame (`sentence_id`, `token`, `annotation`, `prediction`, `start_indices`).

In [12]:
%%time
results_df = wrapped_analyzer.predict_dataset(dataset)
results_df.head()

CPU times: user 1min 51s, sys: 55.7 s, total: 2min 47s
Wall time: 1min 56s


,sentence_id,token,annotation,prediction,start_indices
0,0,The,O,O,0
1,0,address,O,O,4
2,0,of,O,O,12
3,0,Persint,ORGANIZATION,PERSON,15
4,0,is,O,O,23


## 5. Review entity mapping

`CanonicalMapper` auto-resolves entity labels by comparing the `annotation` and `prediction` columns in the results DataFrame. Run the cell below to see the auto-resolved mapping.

If any labels look wrong, update the overrides in the adjustment cell and re-run.

In [13]:
# Create the mapper — labels are extracted and auto-resolved later by get_mapped_results_dataframe()
mapper = CanonicalMapper()

In [24]:
# Auto-resolve labels from annotation + prediction columns and inspect the result
mapped_df = mapper.get_mapped_results_dataframe(results_df, hierarchy=2)
mapper.render_html()
mapped_df.head()

[COUNTRY-FALLBACK] US_BANK_NUMBER -> NATIONAL_ID  ⚠ document type not recognized


Raw Label,Tier,Canonical,Score
US_BANK_NUMBER,⚠ COUNTRY_FALLBACK,NATIONAL_ID,—
ORGANIZATION,EXACT,ORGANIZATION,—
STREET_ADDRESS,EXACT,LOCATION,—
PERSON,EXACT,PERSON,—
DATE_TIME,EXACT,DATE_TIME,—
GPE,EXACT,LOCATION,—
CREDIT_CARD,EXACT,FINANCIAL_PII,—
US_SSN,EXACT,GOVERNMENT_ID,—
US_DRIVER_LICENSE,EXACT,GOVERNMENT_ID,—
AGE,EXACT,DEMOGRAPHIC,—


,sentence_id,token,annotation,prediction,start_indices
0,0,The,O,O,0
1,0,address,O,O,4
2,0,of,O,O,12
3,0,Persint,ORGANIZATION,PERSON,15
4,0,is,O,O,23


In [33]:
# Update mappings, as some aren't tagged in the dataset
mapper.map({"EDUCATION_LEVEL": None,
            "OCCUPATION": None,
            "LICENSE_PLATE": None,
            })

CanonicalMapper(28 resolved, 0 pending)

## 6. Evaluate

In [34]:
# Set up the experiment tracker and log model + dataset params
experiment = get_experiment_tracker()
params = {"dataset_name": dataset_name, "model_name": wrapped_analyzer.name}
params.update(wrapped_analyzer.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(mapper.get_mapping()))

evaluator = SpanEvaluator(iou_threshold=0.75)

In [35]:
results = evaluator.calculate_score_on_df(results_df=mapped_df)

# Global PII precision/recall/F (all entity types collapsed to PII vs O)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, labels=entities)
experiment.end()

# Note that the experiment params and metrics are saved locally


saving experiment data to /Users/omrimendels/Documents/github/presidio-research/notebooks/experiment_20260326-001718.json


## 7. Evaluate results

In [36]:
# Plot output
plotter = Plotter(
    results=results, model_name=wrapped_analyzer.name, display_mode="interactive", beta=2
)
plotter.plot_scores()

In [37]:
pprint(
    {
        "PII F": results.pii_f,
        "PII recall": results.pii_recall,
        "PII precision": results.pii_precision,
    }
)

{'PII F': 0.9044393196886712,
 'PII precision': 0.9054834054834054,
 'PII recall': 0.904178674351585}


## 8. Error analysis

The `ModelError` class groups similar errors together, finding all the false positive and false negative predictions where the model's behavior is consistent. 

This can help in detecting patterns in the model's performance, such as:
- Consistent labeling of specific words or patterns  
- Systematic overprediction or underprediction

**Note:** All errors are displayed using the **mapped model entity types** (e.g., `LOCATION`, `PERSON`) rather than the original dataset entity types (e.g., `STREET_ADDRESS`, `GPE`). This ensures consistency with how the model was evaluated.

In [38]:
plotter.plot_confusion_matrix(entities=entities, confmatrix=confmatrix)

In [39]:
plotter.plot_most_common_tokens()

### 8a. False positives
#### Most common false positive tokens:

In [32]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('royal', 11),
 ('engineers', 10),
 ('street_name', 10),
 ('salesperson', 9),
 ('8 16', 9),
 ('breakfast', 9),
 ('greenlander', 9),
 ('fuse', 8),
 ('zombie', 8),
 ('executives', 7)]
---------------
Example sentence with each FP token:
	- Royal (`royal` pred as PERSON)
	- engineers (`engineers` pred as EMPLOYMENT)
	- { street_name (`street_name` pred as LOCATION)
	- salesperson (`salesperson` pred as EMPLOYMENT)
	- 8 16 (`8 16` pred as DEMOGRAPHIC)
	- breakfast (`breakfast` pred as DATE_TIME)
	- Greenlander (`greenlander` pred as DEMOGRAPHIC)
	- Fuse (`fuse` pred as ORGANIZATION)
	- Zombie (`zombie` pred as EMPLOYMENT)
	- executives (`executives` pred as EMPLOYMENT)


[('royal', 11),
 ('engineers', 10),
 ('street_name', 10),
 ('salesperson', 9),
 ('8 16', 9),
 ('breakfast', 9),
 ('greenlander', 9),
 ('fuse', 8),
 ('zombie', 8),
 ('executives', 7)]

#### More FP analysis

In [ ]:
fps_df = ModelError.get_fps_dataframe(results.model_errors, entity="AGE")
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

### 8b. False negatives (FN)

#### Most common false negative examples + a few samples with FN

In [ ]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

#### More FN analysis

In [ ]:
fns_df = ModelError.get_fns_dataframe(results.model_errors, entity="PERSON")

In [ ]:
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)